# Model-only inference

???????????????????????????????; ?????????????????????????? test ???? compatibility score ??? confidence ???????????????? calibration.


In [ ]:
from pathlib import Path
import json, os, sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from scripts.model_inference import FurniturePredictor, input_items, write_results


## ????? checkpoint

??? selected checkpoint ???????????; ??????????????? completed screening checkpoint ?????????????????? ???????????????????????


In [ ]:
selection = ROOT / 'artifacts/model-only-release/selection-frozen.json'
portable = ROOT / 'release/checkpoint/best.pt'
if portable.is_file():
    checkpoint = portable
    print('Packaged selected checkpoint')
elif selection.is_file():
    checkpoint = Path(json.loads(selection.read_text(encoding='utf-8'))['selected']['run_dir']) / 'best.pt'
    print('Final selection')
else:
    suite = json.loads((ROOT / 'artifacts/runs/suites/furniture-v2-screening.json').read_text(encoding='utf-8'))
    completed = [x for x in suite['runs'] if x['status'] == 'completed']
    if not completed:
        raise FileNotFoundError('???????? completed checkpoint; ????? path ??? checkpoint ????????')
    checkpoint = Path(completed[0]['attempts'][-1]['run_dir']) / 'best.pt'
    print('DEMONSTRATION ONLY: screening checkpoint, not the final selection')
predictor = FurniturePredictor(checkpoint, device='cpu')  # ??????????? cuda ??????????????????
print('checkpoint:', checkpoint, '| device:', predictor.device)


## ????????

????? `image_path` ????????????????? ????? dataset v2 ?????? workspace ???????????????? train; ZIP ?????? dataset ??????????????????????


In [ ]:
image_path = Path('PATH_TO_IMAGE.jpg')  # ????????????????????????
if not image_path.is_file():
    examples = input_items(csv_file=ROOT / 'data/furniture-v2-conservative/train.csv')
    items = [x for x in examples if x[1].is_file()][:3]
    if not items:
        raise FileNotFoundError('????? image_path ???????????: ZIP ???????? dataset ????')
else:
    items = [(image_path.stem, image_path)]
results = predictor.predict(items, top_k=3)
results


?????? output ????????????? ??????????????????????????????????????


In [ ]:
# write_results('artifacts/example-predictions.json', results)
